In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
# SparkSession is the unified entry point (includes SparkContext)
spark = (
    SparkSession.builder.appName("SparkDemo")
    .master("local[*]")
    .config("spark.executor.instances", "2")
    .config("spark.executor.memory", "2g")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

25/11/24 10:01:27 WARN Utils: Your hostname, Tonys-MacBook-Air-2.local resolves to a loopback address: 127.0.0.1, but we couldn't find any external IP address!
25/11/24 10:01:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/24 10:01:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# Data source: https://www.kaggle.com/datasets/yasserh/nyc-taxi-trip-duration
# Create a `data` directory at the root of the project
# Download and place it in the `data` directory

zip_path = "./data/NYC.csv"

trips = spark.read.option("header", "true").option("inferSchema", "true").csv(zip_path)
trips.printSchema()

root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- trip_duration: integer (nullable = true)



In [4]:
trips.show(3)

+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|       id|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|id2875421|        2|2016-03-14 17:24:55|2016-03-14 17:32:30|              1| -73.9821548461914| 40.76793670654297|-73.96463012695312|40.765602111816406|                 N|          455|
|id2377394|        1|2016-06-12 00:43:35|2016-06-12 00:54:38|              1|-73.98041534423828|40.738563537597656|-73.99948120117188| 40.73115158081055|                 N|          663|
|id3858529|        2|2016-01-19 11:35:24|2016-01-19 12:10:48|    

In [5]:
# DSL query – ...
most_passengers = (
    trips.groupBy("vendor_id")
    .agg(F.sum("passenger_count").alias("total_passengers"))
    .orderBy("total_passengers", ascending=False)
)
most_passengers.show()

+---------+----------------+
|vendor_id|total_passengers|
+---------+----------------+
|        2|         1573651|
|        1|          854305|
+---------+----------------+



In [6]:
most_passengers.explain(True)

== Parsed Logical Plan ==
'Sort ['total_passengers DESC NULLS LAST], true
+- Aggregate [vendor_id#18], [vendor_id#18, sum(passenger_count#21) AS total_passengers#108L]
   +- Relation [id#17,vendor_id#18,pickup_datetime#19,dropoff_datetime#20,passenger_count#21,pickup_longitude#22,pickup_latitude#23,dropoff_longitude#24,dropoff_latitude#25,store_and_fwd_flag#26,trip_duration#27] csv

== Analyzed Logical Plan ==
vendor_id: int, total_passengers: bigint
Sort [total_passengers#108L DESC NULLS LAST], true
+- Aggregate [vendor_id#18], [vendor_id#18, sum(passenger_count#21) AS total_passengers#108L]
   +- Relation [id#17,vendor_id#18,pickup_datetime#19,dropoff_datetime#20,passenger_count#21,pickup_longitude#22,pickup_latitude#23,dropoff_longitude#24,dropoff_latitude#25,store_and_fwd_flag#26,trip_duration#27] csv

== Optimized Logical Plan ==
Sort [total_passengers#108L DESC NULLS LAST], true
+- Aggregate [vendor_id#18], [vendor_id#18, sum(passenger_count#21) AS total_passengers#108L]
   +- Pr

In [7]:
# Raw SQL query – same result
trips.createOrReplaceTempView("trips")
sql_res = spark.sql(
    """
    SELECT vendor_id, SUM(passenger_count) AS total_passengers
    FROM trips
    GROUP BY vendor_id
    ORDER BY total_passengers DESC
"""
)
sql_res.show(5)

+---------+----------------+
|vendor_id|total_passengers|
+---------+----------------+
|        2|         1573651|
|        1|          854305|
+---------+----------------+



In [ ]:
# List tables in the current catalog
spark.catalog.listTables()

That was simple enough. There was no need for an external library (like Spark SQL). \
Let's delve a little deeper with a more complex example

In [8]:
result = spark.sql(
    """
    -- 1️⃣ Clean raw rows (remove nulls and absurd coordinates)
    WITH filtered AS (
    SELECT *
    FROM trips
    WHERE   pickup_datetime IS NOT NULL
            AND dropoff_datetime IS NOT NULL
            AND pickup_longitude BETWEEN -180 AND 180
            AND pickup_latitude  BETWEEN -90  AND 90
            AND dropoff_longitude BETWEEN -180 AND 180
            AND dropoff_latitude  BETWEEN -90  AND 90
    ),

    -- 2️⃣ Enrich with temporal & spatial derived columns
    --    • hour_of_day, day_of_week
    --    • haversine distance (km) between pickup & dropoff
    enriched AS (
    SELECT
        id,
        vendor_id,
        passenger_count,
        pickup_datetime,
        dropoff_datetime,
        store_and_fwd_flag,
        trip_duration,
        -- temporal helpers
        hour(pickup_datetime)          AS pickup_hour,
        date_format(pickup_datetime,'E') AS pickup_dow,
        -- haversine distance (Earth radius = 6371 km)
        6371 * acos(
            cos(radians(pickup_latitude)) *
            cos(radians(dropoff_latitude)) *
            cos(radians(dropoff_longitude) - radians(pickup_longitude)) +
            sin(radians(pickup_latitude)) *
            sin(radians(dropoff_latitude))
            )                         AS trip_distance_km
    FROM filtered
    ),

    -- 3️⃣ Flag “long‑haul” trips (distance > 10 km)
    flagged AS (
    SELECT *,
            CASE WHEN trip_distance_km > 10 THEN 1 ELSE 0 END AS long_haul_flag
    FROM enriched
    )

    -- 4️⃣ Final aggregation
    SELECT
    vendor_id,
    pickup_hour,
    pickup_dow,
    long_haul_flag,
    COUNT(*)                         AS trip_count,
    AVG(trip_duration)               AS avg_duration_sec,
    PERCENTILE_APPROX(trip_duration, 0.5) AS median_duration_sec,
    AVG(trip_distance_km)            AS avg_distance_km,
    PERCENTILE_APPROX(trip_distance_km, 0.5) AS median_distance_km,
    MIN(passenger_count)             AS min_passengers,
    MAX(passenger_count)             AS max_passengers,
    AVG(passenger_count)             AS avg_passengers
    FROM flagged
    GROUP BY
    vendor_id,
    pickup_hour,
    pickup_dow,
    long_haul_flag
    ORDER BY
    vendor_id,
    pickup_hour,
    long_haul_flag DESC
    """
)

In [9]:
result.show(20)

+---------+-----------+----------+--------------+----------+------------------+-------------------+------------------+------------------+--------------+--------------+------------------+
|vendor_id|pickup_hour|pickup_dow|long_haul_flag|trip_count|  avg_duration_sec|median_duration_sec|   avg_distance_km|median_distance_km|min_passengers|max_passengers|    avg_passengers|
+---------+-----------+----------+--------------+----------+------------------+-------------------+------------------+------------------+--------------+--------------+------------------+
|        1|          0|       Thu|             1|       203|1696.9014778325122|               1662|15.635184099942332| 13.54871795526582|             1|             4|1.2857142857142858|
|        1|          0|       Tue|             1|       206|11094.252427184467|               1617|15.795135358155397|  14.9581141412621|             1|             4|1.2815533980582525|
|        1|          0|       Fri|             1|       302|1823.

In [10]:
# 1️⃣ Initial filter (nulls & out‑of‑range coordinates)
df_clean = spark.table("trips").filter(
    (F.col("pickup_datetime").isNotNull())
    & (F.col("dropoff_datetime").isNotNull())
    & (F.col("pickup_longitude").between(-180, 180))
    & (F.col("pickup_latitude").between(-90, 90))
    & (F.col("dropoff_longitude").between(-180, 180))
    & (F.col("dropoff_latitude").between(-90, 90))
)


# 2️⃣ Derive temporal columns
df_time = df_clean.withColumn("pickup_hour", F.hour("pickup_datetime")).withColumn(
    "pickup_dow", F.date_format("pickup_datetime", "E")
)

# 3️⃣ Haversine distance (no UDF – pure Spark SQL functions)
R = 6371.0  # Earth radius in km
to_rad = F.radians

df_geo = df_time.withColumn(
    "trip_distance_km",
    R
    * F.acos(
        F.cos(to_rad(F.col("pickup_latitude")))
        * F.cos(to_rad(F.col("dropoff_latitude")))
        * F.cos(to_rad(F.col("dropoff_longitude")) - to_rad(F.col("pickup_longitude")))
        + F.sin(to_rad(F.col("pickup_latitude")))
        * F.sin(to_rad(F.col("dropoff_latitude")))
    ),
)

# 4️⃣ Long‑haul flag (distance > 10 km)

df_flag = df_geo.withColumn(
    "long_haul_flag",
    F.when(F.col("trip_distance_km") > 10, F.lit(1)).otherwise(F.lit(0)),
)

# 5️⃣ Aggregation
agg_df = (
    df_flag.groupBy("vendor_id", "pickup_hour", "pickup_dow", "long_haul_flag")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("trip_duration").alias("avg_duration_sec"),
        F.expr("percentile_approx(trip_duration, 0.5)").alias("median_duration_sec"),
        F.avg("trip_distance_km").alias("avg_distance_km"),
        F.expr("percentile_approx(trip_distance_km, 0.5)").alias("median_distance_km"),
        F.min("passenger_count").alias("min_passengers"),
        F.max("passenger_count").alias("max_passengers"),
        F.avg("passenger_count").alias("avg_passengers"),
    )
    .orderBy("vendor_id", "pickup_hour", F.desc("long_haul_flag"))
)

In [11]:
agg_df.show(truncate=False)

+---------+-----------+----------+--------------+----------+------------------+-------------------+------------------+------------------+--------------+--------------+------------------+
|vendor_id|pickup_hour|pickup_dow|long_haul_flag|trip_count|avg_duration_sec  |median_duration_sec|avg_distance_km   |median_distance_km|min_passengers|max_passengers|avg_passengers    |
+---------+-----------+----------+--------------+----------+------------------+-------------------+------------------+------------------+--------------+--------------+------------------+
|1        |0          |Sun       |1             |261       |1883.3141762452108|1744               |14.258910247986766|13.073668454359865|1             |4             |1.3793103448275863|
|1        |0          |Tue       |1             |206       |11094.252427184467|1617               |15.795135358155397|14.9581141412621  |1             |4             |1.2815533980582525|
|1        |0          |Thu       |1             |203       |1696.